In [ ]:
%%capture
%pip install pandas

In [ ]:
import pandas as pd
import re

In [ ]:
raw_data_path = "data/raw.csv"
raw = pd.read_csv(raw_data_path).drop(columns=['Reactions'])

In [ ]:
raw.head()

In [ ]:
len(raw)

### Remove users you don't want to be modeled (bots, low activitiy)

In [ ]:
# print out all users and their relative activity
raw["Author"].value_counts()

In [ ]:
# choose users to remove
remove_users = [
    "Daily Word Wheel#2230",
    "Black Phone 2 Puzzle#1722",
    "Groot#7975",
    "Wordle#2092",
    "chud#8373",
    "ttimy",
    "seanj10101",
    "is.a",
    "garlicbread6343",
    "Maki#4920",
    "ramen.alt",
    "silva surfer#4104",
]

In [ ]:
# remove users
clean = raw[~raw["Author"].isin(remove_users)]
len(clean)

In [ ]:
# print out all users and their relative activity
clean["Author"].value_counts()

### Add a `has_media` flag for message with media

In [ ]:
URL_PATTERN = r"https?://\S+"

clean["has_media"] = (
    (clean["Attachments"].notna()) & (clean["Attachments"] != "") # has attachments in the Attachments column
) | ( # OR
    clean["Content"].str.contains(URL_PATTERN, case=False, na=False) # has URLs in the Content column
)
clean = clean.drop(columns=["Attachments"])

In [ ]:
clean.head()

### Add a `has_text` flag for message with content (for message with media AND content)

In [ ]:
clean["has_text"] = (
    clean["Content"]
    .fillna("")
    .str.replace(URL_PATTERN, "", regex=True) # remove URLs
    .str.strip()
    .str.len() > 0
)

In [ ]:
clean.head()

### Change from alias and nicknames to discord username

Fill in the map below with all aliases and nicknames of each user you want in the GNN.

In [ ]:
# nickname/server name to discord username mapping
nickname_to_username = {
    "nathan": "n.atan",
    "jacob": "jab.bo",
    "fatcob": "jab.bo",
    "jabbo": "jab.bo",
    "fatcob": "jab.bo",
    "glenn": "glxnn_",
    "grack": "grack6097",
    "grace": "grack6097",
    "dawid": "dawid7701",
    "david": "dawid7701",
    "fatvid": "dawid7701",
    "squigglyu": "will_i_am_123",
    "william": "will_i_am_123",
    "will": "will_i_am_123",
    "pigcob": "heehooo",
    "joshua": "heehooo",
    "josh": "heehooo",
    "bagel": "mundies",
    "sean": "mundies",
    "victim": "dawneel",
    "daniel": "dawneel",
    "dan": "dawneel",
    "ramen": "yramen",
    "kevin": "yramen",
    "fatvin": "yramen",
    "lanaboglenn": "lanabobananas",
    "allan": "lanabobananas",
    "alan": "lanabobananas",
}

In [ ]:
pattern = re.compile(
    r"@?\b(" + "|".join(map(re.escape, nickname_to_username.keys())) + r")\b",
    flags=re.IGNORECASE
)
pattern

In [ ]:
clean["Content"] = clean["Content"].fillna("").str.replace(
    pattern,
    lambda m: ("@" if m.group(0).startswith("@") else "") + nickname_to_username[m.group(1).lower()],
    regex=True
)

In [ ]:
clean.head()

### Save cleaned dataset for graph creation

In [ ]:
clean.to_csv("data/clean.csv")